[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S16/2_Practica/en_clase/S16_01_demostrador_en_clase.ipynb)

# S16 · 01 — Plantilla de demostrador · versión en clase

⏱ 45 min en clase · Sprint 16 · Proyecto: evaluación, IA responsable y entrega (píldora de apoyo)

**Contexto TurisData.** La rúbrica pide un **demostrador ejecutable** (requisito R7) y que sea **reproducible desde el README** (criterio F). Un demostrador no es "el notebook con todo": es una herramienta con la que **otra persona** prueba tu modelo sin saber programar, con **avisos de incertidumbre** y **errores del usuario bien gestionados**. Aquí construimos una plantilla completa con el modelo de cancelaciones de TurisData que tu equipo adaptará a su proyecto.

**Cuatro opciones de demostrador (elige una, la primera es la recomendada):**

| Opción | Cuándo | Ventaja | Inconveniente |
|---|---|---|---|
| **Streamlit** (`app.py`) | Modelos tabulares, series, texto | Web interactiva con pocas líneas | Hay que instalarlo y lanzar `streamlit run` |
| **Gradio** | Imágenes, audio, chat (Hugging Face) | Muy rápido para subir una foto o escribir | Menos control del diseño |
| **Notebook con widgets** | Sin instalar nada / como plan B | Funciona en Colab | Menos "producto" |
| **API con FastAPI** (opcional) | Si otro sistema debe llamar al modelo | Estándar en producción | Requiere cliente que la consuma |

En este cuaderno construimos tres (Streamlit, widgets y FastAPI). Gradio no tiene sección propia: úsalo si tu proyecto trabaja con imágenes, audio o chat.

**Este cuaderno no tiene ejercicios**: en el aula solo ejecutas y lees los ejemplos. Los 3 ejercicios (coherencia entre campos, mensaje de errores y pruebas con dos reservas), el panel con widgets, la API con FastAPI (opcional), la lista de reproducibilidad, el mini-reto para tu proyecto y las preguntas de reflexión están en el cuaderno completo.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, json, py_compile
from datetime import date
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEMILLA = 42
np.random.seed(SEMILLA)
CARPETA = "demostrador"                    # aquí se crean modelo/, logica.py, app.py y api.py
os.makedirs(os.path.join(CARPETA, "modelo"), exist_ok=True)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Entrenar el modelo de cancelaciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué AUC obtiene el modelo en el test?
2. ¿Qué significa un AUC de 0,5?
3. ¿Por qué el `Pipeline` guarda juntos el preprocesado y el modelo?

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
NUMERICAS = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche", "cliente_repetidor", "cancelaciones_previas"]
CATEGORICAS = ["tipo_habitacion", "canal", "tarifa", "deposito"]
X, y = reservas[NUMERICAS + CATEGORICAS], reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

prep = ColumnTransformer([("num", StandardScaler(), NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
modelo = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)
auc_test = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
print(f"AUC en test: {auc_test:.3f}")

### 👀 A3 · EJEMPLO — Guardar el modelo y los metadatos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué dos archivos se guardan y para qué sirve cada uno?
2. ¿Por qué guardar las versiones de las librerías?
3. ¿Qué campo avisa del uso que NO debe hacerse del modelo?

In [ ]:
meta = {
    "nombre": "Riesgo de cancelación de reservas · TurisData (datos sintéticos)",
    "campos": {
        "antelacion_dias": {"tipo": "entero", "min": int(X["antelacion_dias"].min()), "max": int(X["antelacion_dias"].max()), "etiqueta": "Antelación (días entre reserva y llegada)"},
        "noches": {"tipo": "entero", "min": 1, "max": 30, "etiqueta": "Noches"},
        "adultos": {"tipo": "entero", "min": 1, "max": 6, "etiqueta": "Adultos"},
        "ninos": {"tipo": "entero", "min": 0, "max": 6, "etiqueta": "Niños"},
        "precio_noche": {"tipo": "decimal", "min": 20.0, "max": 600.0, "etiqueta": "Precio por noche (€)"},
        "cliente_repetidor": {"tipo": "entero", "min": 0, "max": 1, "etiqueta": "Cliente repetidor (0 = no, 1 = sí)"},
        "cancelaciones_previas": {"tipo": "entero", "min": 0, "max": 10, "etiqueta": "Cancelaciones previas"},
        **{c: {"tipo": "categoria", "valores": sorted(X[c].unique().tolist()), "etiqueta": c.replace("_", " ").capitalize()} for c in CATEGORICAS},
    },
    "auc_test": round(float(auc_test), 3),
    "n_entrenamiento": int(len(X_ent)),
    "tasa_cancelacion_base": round(float(y.mean()), 3),
    "entrenado": str(date.today()),
    "semilla": SEMILLA,
    "versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "limitaciones": "Entrenado con datos sintéticos de un solo alojamiento ficticio. No usar para decidir sobre personas concretas: solo orienta la gestión de disponibilidad.",
}
joblib.dump(modelo, os.path.join(CARPETA, "modelo", "modelo_cancelaciones.joblib"))
with open(os.path.join(CARPETA, "modelo", "metadatos.json"), "w", encoding="utf8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Guardado:", os.listdir(os.path.join(CARPETA, "modelo")))

### 👀 A4 · EJEMPLO — Escribir logica.py y probarla

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué probabilidad de cancelación y qué nivel sale para `reserva_ejemplo`?
2. ¿Qué devuelve `predecir_reserva` si la entrada es errónea?
3. ¿Por qué `logica.py` no importa Streamlit?

In [ ]:
LOGICA = r'''"""Lógica del demostrador: cargar el modelo, validar la entrada y predecir. Sin interfaz."""
import json
import os

import joblib
import pandas as pd

CARPETA = os.path.dirname(os.path.abspath(__file__))


def cargar_modelo(carpeta=None):
    """Devuelve (modelo, metadatos). Lanza FileNotFoundError con un mensaje claro si faltan los archivos."""
    carpeta = carpeta or os.path.join(CARPETA, "modelo")
    ruta_modelo = os.path.join(carpeta, "modelo_cancelaciones.joblib")
    ruta_meta = os.path.join(carpeta, "metadatos.json")
    if not (os.path.exists(ruta_modelo) and os.path.exists(ruta_meta)):
        raise FileNotFoundError("Falta el modelo. Ejecuta primero el notebook S16_01 (o tu script de entrenamiento) para crear la carpeta 'modelo/'.")
    with open(ruta_meta, encoding="utf8") as f:
        return joblib.load(ruta_modelo), json.load(f)


def validar_reserva(reserva, meta):
    """Devuelve una lista de errores en español (lista vacía = entrada válida)."""
    errores = []
    for campo, regla in meta["campos"].items():
        if campo not in reserva or reserva[campo] in (None, ""):
            errores.append(f"Falta el campo '{campo}'.")
            continue
        valor = reserva[campo]
        if regla["tipo"] == "categoria":
            if valor not in regla["valores"]:
                errores.append(f"'{valor}' no es un valor válido para '{campo}'. Opciones: {', '.join(regla['valores'])}.")
        else:
            try:
                numero = float(valor)
            except (TypeError, ValueError):
                errores.append(f"'{campo}' debe ser un número (has escrito '{valor}').")
                continue
            if not regla["min"] <= numero <= regla["max"]:
                errores.append(f"'{campo}' debe estar entre {regla['min']} y {regla['max']} (has escrito {valor}).")
    return errores


def nivel_riesgo(probabilidad):
    """Traduce la probabilidad a una etiqueta que entiende quien no sabe de modelos."""
    if probabilidad < 0.30:
        return "bajo"
    if probabilidad < 0.50:
        return "medio"
    return "alto"


def predecir_reserva(reserva, modelo, meta):
    """Valida y predice. Devuelve un diccionario; si hay errores, solo {'errores': [...]}."""
    errores = validar_reserva(reserva, meta)
    if errores:
        return {"errores": errores}
    fila = pd.DataFrame([reserva])[list(meta["campos"])]
    probabilidad = float(modelo.predict_proba(fila)[0, 1])
    return {
        "errores": [],
        "probabilidad": round(probabilidad, 3),
        "nivel": nivel_riesgo(probabilidad),
        "tasa_base": meta["tasa_cancelacion_base"],
        "aviso": (f"Estimación orientativa. El modelo tiene un AUC de {meta['auc_test']} en datos de prueba: "
                  "se equivoca con frecuencia. "
                  + meta["limitaciones"]),
    }
'''
with open(os.path.join(CARPETA, "logica.py"), "w", encoding="utf8") as f:
    f.write(LOGICA)

sys.path.insert(0, CARPETA)
import importlib, logica
importlib.reload(logica)
modelo_cargado, meta_cargado = logica.cargar_modelo()

reserva_ejemplo = {"antelacion_dias": 120, "noches": 5, "adultos": 2, "ninos": 1, "precio_noche": 95.0, "cliente_repetidor": 0,
                   "cancelaciones_previas": 0, "tipo_habitacion": "estandar", "canal": "web_intermediaria", "tarifa": "flexible", "deposito": "sin_deposito"}
print(logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado))
print(logica.predecir_reserva({**reserva_ejemplo, "noches": -3, "canal": "telepatia"}, modelo_cargado, meta_cargado))

### 👀 A5 · EJEMPLO — Dos reservas de comparación (ya hecho)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué riesgo sale para cada reserva?
2. ¿Qué campos se cambian para pasar de una a otra?

In [ ]:
reserva_riesgo_alto = {**reserva_ejemplo, "antelacion_dias": 250, "deposito": "sin_deposito", "tarifa": "flexible", "canal": "web_intermediaria"}
reserva_riesgo_bajo = {**reserva_ejemplo, "antelacion_dias": 7, "deposito": "total", "tarifa": "no_reembolsable", "canal": "directo"}
p_alto = logica.predecir_reserva(reserva_riesgo_alto, modelo_cargado, meta_cargado)["probabilidad"]
p_bajo = logica.predecir_reserva(reserva_riesgo_bajo, modelo_cargado, meta_cargado)["probabilidad"]
print(f"Riesgo alto {p_alto:.0%} · riesgo bajo {p_bajo:.0%}")

### 👀 A6 · EJEMPLO — Pruebas de sentido común (las de tests/test_logica.py)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántas pruebas pasan y cuál comprueba el sentido común?
2. ¿Qué comprueba `test_entrada_invalida_no_rompe`?
3. ¿Con qué comando se ejecutarían desde la terminal?

In [ ]:
# Estas pruebas se guardarían tal cual en tests/test_logica.py (se ejecutan con:  pytest -q)
def test_reserva_valida_devuelve_probabilidad():
    r = logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado)
    assert 0 <= r["probabilidad"] <= 1 and r["nivel"] in ("bajo", "medio", "alto") and "aviso" in r

def test_entrada_invalida_no_rompe():
    r = logica.predecir_reserva({**reserva_ejemplo, "noches": -3}, modelo_cargado, meta_cargado)
    assert r["errores"] and "probabilidad" not in r

def test_falta_un_campo():
    incompleta = {k: v for k, v in reserva_ejemplo.items() if k != "canal"}
    assert any("canal" in e for e in logica.validar_reserva(incompleta, meta_cargado))

def test_sentido_comun():
    assert p_alto > p_bajo

for prueba in (test_reserva_valida_devuelve_probabilidad, test_entrada_invalida_no_rompe, test_falta_un_campo, test_sentido_comun):
    prueba()
    print("OK ", prueba.__name__)

### 👀 A7 · EJEMPLO — Generar app.py (Streamlit)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Se ejecuta la app dentro del notebook?
2. ¿Qué muestra la app cuando la entrada es válida?
3. ¿Dónde está la lógica de validar y predecir?

In [ ]:
APP = r'''"""Demostrador Streamlit · TurisData. Lanzar con:  streamlit run app.py"""
import streamlit as st

from logica import cargar_modelo, predecir_reserva

st.set_page_config(page_title="Riesgo de cancelación · TurisData", layout="centered")


@st.cache_resource
def cargar():
    return cargar_modelo()


st.title("Riesgo de cancelación de una reserva")
st.caption("Demostrador del proyecto. Datos sintéticos de TurisData Canarias (empresa ficticia).")

try:
    modelo, meta = cargar()
except FileNotFoundError as error:
    st.error(str(error))
    st.stop()

with st.form("formulario"):
    reserva = {}
    columnas = st.columns(2)
    for i, (campo, regla) in enumerate(meta["campos"].items()):
        with columnas[i % 2]:
            if regla["tipo"] == "categoria":
                reserva[campo] = st.selectbox(regla["etiqueta"], regla["valores"])
            elif regla["tipo"] == "entero":
                reserva[campo] = st.number_input(regla["etiqueta"], value=int(max(regla["min"], min(regla["max"], 1))), step=1)
            else:
                reserva[campo] = st.number_input(regla["etiqueta"], value=float(regla["min"] + 75.0), step=5.0)
    enviado = st.form_submit_button("Calcular riesgo")

if enviado:
    resultado = predecir_reserva(reserva, modelo, meta)
    if resultado["errores"]:
        st.warning("No se puede calcular todavía:\n\n" + "\n".join(f"- {e}" for e in resultado["errores"]))
    else:
        c1, c2 = st.columns(2)
        c1.metric("Probabilidad de cancelación", f"{resultado['probabilidad']:.0%}", f"{resultado['probabilidad'] - resultado['tasa_base']:+.0%} frente a la media")
        c2.metric("Nivel de riesgo", resultado["nivel"].upper())
        st.progress(resultado["probabilidad"])
        st.info(resultado["aviso"])
'''
with open(os.path.join(CARPETA, "app.py"), "w", encoding="utf8") as f:
    f.write(APP)
py_compile.compile(os.path.join(CARPETA, "app.py"), doraise=True)      # comprueba que no tiene errores de sintaxis
print("app.py creado y sin errores de sintaxis. Para verlo: cd demostrador && streamlit run app.py")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S16/2_Practica/para_casa/S16_01_demostrador.ipynb)** trae los 3 ejercicios, el panel con widgets, la API con FastAPI (opcional), la lista de reproducibilidad, el mini-reto para tu proyecto y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** La rúbrica pide un **demostrador ejecutable** (requisito R7) que otra persona pueda lanzar solo con el README (criterio F). Aquí construimos una plantilla completa con el modelo de cancelaciones de TurisData, que tu equipo adaptará a su proyecto.

- **Qué hace:**
    - Define `dato(...)`, que localiza (o descarga) los archivos de datos del curso.
    - Carga `numpy`, `pandas`, `joblib` y de `scikit-learn` lo necesario para entrenar una regresión logística con su preprocesado.
    - Crea la carpeta `demostrador/modelo/`, donde se guardará todo lo que genera este cuaderno.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Guardar el modelo y sus metadatos

#### A2 · Ejemplo — Entrenar el modelo de cancelaciones

**Idea.** El demostrador **no reentrena**: carga un modelo ya entrenado. Guardamos junto al modelo un
archivo `metadatos.json` con todo lo que la interfaz necesita saber: qué campos pedir, qué rangos son
válidos, qué categorías existen, con qué datos y versiones se entrenó y **cuánto vale el modelo**
(así el usuario ve sus límites). El **pipeline** (preprocesado + modelo) se guarda entero con `joblib`,
para que la app reciba una fila "cruda" igual que en el entrenamiento.

**Ejemplo resuelto:** entrenamos una regresión logística con 11 variables sencillas.

- **Qué hace:**
    - Carga las reservas (`reservas_turisdata.csv`) y separa las 7 variables numéricas y las 4 categóricas.
    - Reserva el 20 % para el test (`train_test_split`, con `stratify=y` para mantener la proporción de canceladas).
    - Monta un `Pipeline`: el preprocesado (escalar números y codificar categorías) y la regresión logística, en una sola pieza.
    - Calcula el AUC en el test (`auc_test`).
- **Qué debe salir:**
    - `AUC en test: 0.746` (aproximadamente: puede variar en el tercer decimal según la versión de scikit-learn).
- **Cómo leerlo:**
    - El AUC mide cómo de bien ordena el modelo las reservas: 0,5 es el azar y 1 sería perfecto. **0,75 es un modelo modesto**: acierta bastante más que el azar, pero se equivoca con frecuencia. Por eso más adelante el demostrador lo avisa en cada respuesta.
- **Si falla:**
    - `NameError: name 'dato' is not defined` o `'pd'` → falta la celda de preparación.
    - `FileNotFoundError` / error de descarga de `reservas_turisdata.csv` → comprueba la conexión (la primera vez se descarga).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC obtiene el modelo en el test?* → Alrededor de **0,75** (0,746).
2. *¿Qué significa un AUC de 0,5?* → Que ordena como el **azar**: no aporta nada.
3. *¿Por qué el `Pipeline` guarda juntos el preprocesado y el modelo?* → Para que la app reciba una **fila cruda**, igual que en el entrenamiento, sin repetir el preprocesado a mano.

</details>

#### A3 · Ejemplo — Guardar el modelo y los metadatos

- **Qué hace:**
    - `meta` reúne lo que la interfaz necesita saber: los **campos** a pedir (tipo, mínimo, máximo y etiqueta de cada uno; las categorías con sus valores), el AUC, el tamaño del entrenamiento, la tasa de cancelación base, la fecha, la semilla, las **versiones** de las librerías y las **limitaciones**.
    - `joblib.dump` guarda el pipeline entero en `modelo/modelo_cancelaciones.joblib`.
    - `json.dump` escribe `modelo/metadatos.json` (legible por personas).
- **Qué debe salir:**
    - `Guardado: ['modelo_cancelaciones.joblib', 'metadatos.json']` (los dos archivos, en cualquier orden).
- **Cómo leerlo:**
    - El demostrador **no reentrena**: carga estos dos archivos. Los metadatos permiten que la app construya el formulario y que el usuario vea los **límites** del modelo.
- **Si falla:**
    - `NameError: name 'modelo' is not defined` → falta ejecutar la celda anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos archivos se guardan y para qué sirve cada uno?* → `modelo_cancelaciones.joblib` es el **modelo** (pipeline entero); `metadatos.json` dice qué **campos pedir**, qué **rangos** son válidos y cuáles son los **límites** del modelo.
2. *¿Por qué guardar las versiones de las librerías?* → Para poder **reproducir** el resultado: otra versión puede dar otro modelo.
3. *¿Qué campo avisa del uso que NO debe hacerse del modelo?* → `limitaciones`.

</details>

## 2. La lógica, separada de la interfaz

#### A4 · Ejemplo — Escribir logica.py y probarla

**Idea.** Si validar y predecir viven **dentro** de la app, no se pueden probar ni reutilizar en el
notebook o en la API. Ponemos esa lógica en un archivo `logica.py` **sin nada de Streamlit**. La app,
los widgets y la API serán capas finas que solo llaman a `logica.py`.

**Ejemplo resuelto:** escribimos `logica.py`. Léelo con calma: es lo que adaptará tu equipo.

- **Qué hace:**
    - `LOGICA` es el texto de un archivo `logica.py` con cuatro funciones: `cargar_modelo` (carga o falla con un mensaje claro), `validar_reserva` (lista de errores en español), `nivel_riesgo` (bajo, medio o alto) y `predecir_reserva` (valida, predice y devuelve un diccionario).
    - Se escribe en `demostrador/logica.py`, se **importa** como cualquier módulo y se carga el modelo guardado (`modelo_cargado`, `meta_cargado`).
    - `reserva_ejemplo` es una reserva completa con los 11 campos; se predice con ella y con otra errónea (`noches=-3`, `canal="telepatia"`).
- **Qué debe salir:**
    - Dos diccionarios. El primero: `{'errores': [], 'probabilidad': 0.812, 'nivel': 'alto', 'tasa_base': 0.459, 'aviso': 'Estimación orientativa. El modelo tiene un AUC de 0.746…'}`.
    - El segundo, solo con los errores: `{'errores': ["'noches' debe estar entre 1 y 30 (has escrito -3).", "'telepatia' no es un valor válido para 'canal'. Opciones: agencia, directo, empresa, web_intermediaria."]}`.
- **Cómo leerlo:**
    - **Errores en español y accionables:** dicen qué campo falla, qué se esperaba y qué se recibió.
    - **El resultado incluye el aviso de incertidumbre** (AUC, límites, uso no permitido): viaja con cada respuesta.
    - **Devuelve datos, no imprime:** así la misma función sirve a la app, a los widgets y a la API.
- **Si falla:**
    - `FileNotFoundError: Falta el modelo…` → no has ejecutado la celda que guarda el modelo.
    - `NameError: name 'CARPETA' is not defined` → falta la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué probabilidad de cancelación y qué nivel sale para `reserva_ejemplo`?* → **0,812** (≈ 81 %), nivel **alto** (la media histórica es 0,459).
2. *¿Qué devuelve `predecir_reserva` si la entrada es errónea?* → Solo `{'errores': [...]}`, **sin probabilidad**: no se predice con datos malos.
3. *¿Por qué `logica.py` no importa Streamlit?* → Para poder **probarla y reutilizarla** en la app, los widgets y la API.

</details>

## 3. Pruebas de sentido común

#### A5 · Ejemplo — Dos reservas de comparación (ya hecho)

**Idea.** Un modelo puede fallar sin dar error: devolver probabilidades absurdas. Antes de enseñarlo,
escribimos **pruebas** (`assert`) sobre casos cuyo resultado esperamos: *"una reserva con riesgo alto sobre
el papel debe puntuar más que una de riesgo bajo"*. Estas pruebas se guardan en el repositorio
(`tests/test_logica.py`) y suman puntos en el criterio F (pruebas).

- **Qué hace:**
    - `reserva_riesgo_alto` es la reserva de ejemplo con mucha antelación (250 días), sin depósito, tarifa flexible y canal `web_intermediaria`; `reserva_riesgo_bajo`, con poca antelación (7 días), depósito total, tarifa no reembolsable y canal `directo`.
    - Calcula la probabilidad de cada una con `logica.predecir_reserva` (`p_alto` y `p_bajo`).
- **Qué debe salir:**
    - `Riesgo alto 95% · riesgo bajo 4%` (aproximadamente).
- **Cómo leerlo:**
    - Sobre el papel, una reserva barata de cancelar debería puntuar mucho más alto que otra comprometida. Eso es **sentido común**, y las pruebas de la celda siguiente lo comprueban. Este mismo ejercicio lo haces tú en el cuaderno completo (ejercicio 3).
- **Si falla:**
    - `NameError: name 'logica' is not defined` → falta el ejemplo de `logica.py`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué riesgo sale para cada reserva?* → Alrededor de **95 %** para la de riesgo alto y **4 %** para la de riesgo bajo.
2. *¿Qué campos se cambian para pasar de una a otra?* → **Antelación, depósito, tarifa y canal**.

</details>

#### A6 · Ejemplo — Pruebas de sentido común (las de tests/test_logica.py)

- **Qué hace:**
    - Define cuatro funciones de prueba con `assert`: una reserva válida devuelve una probabilidad entre 0 y 1, un nivel permitido y el aviso; una entrada inválida no rompe y devuelve errores sin probabilidad; si falta un campo, el error lo nombra; y la reserva de riesgo alto puntúa más que la de riesgo bajo.
    - Un bucle las ejecuta una a una e imprime `OK` con el nombre de cada una.
- **Qué debe salir:**
    - ```
OK  test_reserva_valida_devuelve_probabilidad
OK  test_entrada_invalida_no_rompe
OK  test_falta_un_campo
OK  test_sentido_comun
```
- **Cómo leerlo:**
    - Estas pruebas se guardarían tal cual en `tests/test_logica.py` y se lanzarían con `pytest -q`: suman puntos en el criterio F (pruebas). Si una falla, salta un `AssertionError` con el nombre de la prueba.
- **Si falla:**
    - `NameError: name 'p_alto' is not defined` → falta la celda con las dos reservas de comparación.
    - `AssertionError` en una prueba → el modelo o `logica.py` han cambiado: revisa la última modificación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas pruebas pasan y cuál comprueba el sentido común?* → **Cuatro**; la de sentido común es `test_sentido_comun` (`p_alto > p_bajo`).
2. *¿Qué comprueba `test_entrada_invalida_no_rompe`?* → Que una reserva errónea devuelve **errores y no probabilidad**, sin lanzar excepción.
3. *¿Con qué comando se ejecutarían desde la terminal?* → `pytest -q`.

</details>

## 4. Opción A (recomendada): app de Streamlit

#### A7 · Ejemplo — Generar app.py (Streamlit)

**Idea.** Streamlit convierte un script de Python en una web. El script se **vuelve a ejecutar de arriba
abajo cada vez que el usuario toca algo**. Solo necesita la lógica de la sección 2.

**Ejemplo resuelto:** generamos `demostrador/app.py`. **No lo ejecutamos aquí**: se lanza desde una terminal.

```bash
pip install streamlit joblib scikit-learn pandas
cd demostrador
streamlit run app.py
```
En Colab, Streamlit no se abre directamente (hace falta un túnel); por eso en Colab usa la **opción B**
(widgets) y deja Streamlit para el equipo en local o en *Streamlit Community Cloud*.

- **Qué hace:**
    - `APP` es el texto de `demostrador/app.py`: importa Streamlit y `cargar_modelo` y `predecir_reserva` de `logica.py`.
    - Carga el modelo una sola vez (`@st.cache_resource`) y, si faltan los archivos, muestra el error y se detiene.
    - Construye un formulario con un campo por cada entrada de `meta["campos"]` (lista desplegable para las categorías, numérico para el resto).
    - Al pulsar «Calcular riesgo»: si hay errores muestra un aviso con ellos; si no, la probabilidad, el nivel de riesgo, una barra de progreso y el **aviso de incertidumbre**.
    - Se guarda y `py_compile` comprueba que no tiene errores de sintaxis. **No se ejecuta aquí**.
- **Qué debe salir:**
    - `app.py creado y sin errores de sintaxis. Para verlo: cd demostrador && streamlit run app.py`.
- **Cómo leerlo:**
    - La app no tiene lógica propia: es una **capa fina** sobre `logica.py`. Para verla funcionar hay que lanzarla desde una terminal (`streamlit run app.py`); en Colab usa la opción B.
- **Si falla:**
    - `NameError: name 'CARPETA' is not defined` → falta la celda de preparación.
    - `py_compile.PyCompileError` → has modificado el texto de `APP` y hay un error de sintaxis; el mensaje indica la línea.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Se ejecuta la app dentro del notebook?* → **No**: solo se genera y se comprueba la sintaxis; se lanza con `streamlit run app.py` desde una terminal.
2. *¿Qué muestra la app cuando la entrada es válida?* → La **probabilidad**, el **nivel de riesgo**, una barra y el **aviso de incertidumbre**.
3. *¿Dónde está la lógica de validar y predecir?* → En **`logica.py`**; la app solo la llama.

</details>